<div>

# **AI agent use Tavily**

<div dir=RTL>

# **התקנות / ספריות**

In [14]:
# התקנה
!pip install -q langgraph langchain-google-genai langchain-community tavily-python
# ספריות
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
# מפתחות
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

<div dir=RTL>

# **הגדרת מודל וכלים**

In [15]:
# כלים
tools = [TavilySearchResults(max_results=3)]

# מודל
model = ChatGoogleGenerativeAI(
    model="gemini-pro-latest",
    google_api_key=os.environ["GOOGLE_API_KEY"]
).bind_tools(tools)

<div dir=RTL>

# **הגדרת גרף**

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# צומת המודל
def call_model(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}

# בדיקה: האם המודל ביקש להפעיל כלי?
def should_continue(state: MessagesState):
    return "tools" if state["messages"][-1].tool_calls else END

# בניית הגרף
graph = StateGraph(MessagesState)
graph.add_node("model", call_model)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_continue)
graph.add_edge("tools", "model")

agent = graph.compile()

# הצגת הגרף
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

<div dir=RTL>

# **הפעלת סוכן**

In [20]:
from langchain_core.messages import HumanMessage

# שאל את הסוכן שאלה
question = "מה החדשות הכי מעניינות בתחום ה-AI היום?"

collected_messages = []
for s in agent.stream({"messages": [HumanMessage(content=question)]}):
    # LangGraph's stream often yields dictionary where keys are node names
    # and values are the output of that node. We need to check these values.
    # The 'messages' are usually nested.

    # Check if 'model' node output messages
    if "model" in s and isinstance(s["model"], dict) and "messages" in s["model"]:
        collected_messages.extend(s["model"]["messages"])
    # Check if 'tools' node output messages (if applicable for debugging/tracing tool calls)
    elif "tools" in s and isinstance(s["tools"], dict) and "messages" in s["tools"]:
        collected_messages.extend(s["tools"]["messages"])
    # If the state itself contains messages (e.g., final state update might just have 'messages')
    elif "messages" in s:
        collected_messages.extend(s["messages"])

result = {"messages": collected_messages}

<div dir=RTL>

# **הדפסת התשובה**

In [21]:
# הדפסת תשובה
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'הנה החדשות והמגמות המעניינות והבולטות ביותר בתחום הבינה המלאכותית נכון להיום (אפריל 2026):\n\n**1. המהפך בצמרת: Anthropic עוקפת את OpenAI**\nעל פי הדיווחים האחרונים, חברת Anthropic (המפתחת של מודלי Claude) רשמה צמיחה חסרת תקדים של פי 30 בתוך 15 חודשים, והגיעה לקצב הכנסות שנתי (ARR) של 30 מיליארד דולר. בכך היא עקפה את OpenAI (העומדת על כ-25 מיליארד דולר). במקביל, Anthropic שחררה לאחרונה את המודל **Claude Opus 4.7**, שמוביל כרגע במרבית מבחני הביצועים (בנצ\'מרקים), למרות שעלויות ההרצה שלו גבוהות יותר.\n\n**2. המעבר לסוכנים אוטונומיים (Agentic AI)**\nהמגמה המרכזית כיום היא המעבר מבינה מלאכותית יוצרת "קלאסית" (שמגיבה לפרומפטים כמו צ\'אטבוט) למערכות של סוכני AI אוטונומיים. אלו אינן רק מערכות להשלמת טקסט, אלא סוכנים שמסוגלים לפעול באופן עצמאי, ליזום, לתכנן מהלכים אסטרטגיים קדימה ולבצע שרשראות של משימות מורכבות לאורך זמן ללא התערבות אנושית. \n\n**3. פריצת דרך דרמטית ביעילות: מודלים של 1-ביט (1-bit LLMs)**\nככל שהמודלים הפכו למורכבים יותר, עלויות החישוב וצריכת האנרגיה